In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
# Source volume path
base_path = "/Volumes/clinical_trials/raw/clinical_trials_raw"

# Target catalog and schema
catalog = "clinical_trials"
bronze_schema = "bronze"

# Tables to ingest
tables = {
    "studies": f"{base_path}/studies.txt",
    "calculated_values": f"{base_path}/calculated_values.txt",
    "conditions": f"{base_path}/conditions.txt",
    "sponsors": f"{base_path}/sponsors.txt",
    "interventions": f"{base_path}/interventions.txt",
    "facilities": f"{base_path}/facilities.txt"
}

print("Configuration loaded.")
print(f"Target: {catalog}.{bronze_schema}")

Configuration loaded.
Target: clinical_trials.bronze


In [0]:
#Ingesting bronze tables
def ingest_to_bronze(table_name, source_path):
    print(f"Ingesting {table_name}...")
    
    df = (spark.read
          .option("header", "true")
          .option("sep", "|")
          .option("multiline", "true")
          .option("escape", '"')
          .csv(source_path))
    
    # Add ingestion metadata columns
    df = (df
          .withColumn("_ingested_at", current_timestamp())
          .withColumn("_source_file", lit(source_path)))
    
    # Write to Bronze Delta table
    (df.write
       .format("delta")
       .mode("overwrite")
       .saveAsTable(f"{catalog}.{bronze_schema}.bronze_{table_name}"))
    
    count = spark.table(f"{catalog}.{bronze_schema}.bronze_{table_name}").count()
    print(f"  ✓ bronze_{table_name}: {count:,} rows")

# Ingest all tables
for table_name, source_path in tables.items():
    ingest_to_bronze(table_name, source_path)

print("\nBronze ingestion complete.")

Ingesting studies...
  ✓ bronze_studies: 587,788 rows
Ingesting calculated_values...
  ✓ bronze_calculated_values: 587,788 rows
Ingesting conditions...
  ✓ bronze_conditions: 1,051,178 rows
Ingesting sponsors...
  ✓ bronze_sponsors: 938,438 rows
Ingesting interventions...
  ✓ bronze_interventions: 993,947 rows
Ingesting facilities...
  ✓ bronze_facilities: 3,461,990 rows

Bronze ingestion complete.


In [0]:
#Verify Delta tables
print("Bronze tables in catalog:")
display(spark.sql(f"SHOW TABLES IN {catalog}.{bronze_schema}"))

Bronze tables in catalog:


database,tableName,isTemporary
bronze,bronze_calculated_values,false
bronze,bronze_conditions,false
bronze,bronze_facilities,false
bronze,bronze_interventions,false
bronze,bronze_sponsors,false
bronze,bronze_studies,false


In [0]:
print("Sample from bronze_studies with metadata columns:")
display(
    spark.table(f"{catalog}.{bronze_schema}.bronze_studies")
    .select("nct_id", "overall_status", "phase", "start_date", "completion_date", "_ingested_at", "_source_file")
    .limit(5)
)

Sample from bronze_studies with metadata columns:


nct_id,overall_status,phase,start_date,completion_date,_ingested_at,_source_file
NCT01442103,COMPLETED,NA,2011-09-30,2012-05-31,2026-06-12T12:21:12.824Z,/Volumes/clinical_trials/raw/clinical_trials_raw/studies.txt
NCT00368147,COMPLETED,null,2002-04-30,2007-02-28,2026-06-12T12:21:12.824Z,/Volumes/clinical_trials/raw/clinical_trials_raw/studies.txt
NCT05259163,COMPLETED,NA,2022-06-07,2023-01-06,2026-06-12T12:21:12.824Z,/Volumes/clinical_trials/raw/clinical_trials_raw/studies.txt
NCT04876820,TERMINATED,NA,2021-10-01,2024-10-01,2026-06-12T12:21:12.824Z,/Volumes/clinical_trials/raw/clinical_trials_raw/studies.txt
NCT05130606,RECRUITING,NA,2021-06-01,2025-12-31,2026-06-12T12:21:12.824Z,/Volumes/clinical_trials/raw/clinical_trials_raw/studies.txt
